In [19]:
import boto3
import boto3
import duckdb
import tempfile
from datetime import datetime
import json

In [12]:
def extract_timestamp(folder_name):
    timestamp_str = folder_name.rstrip("/").replace("ingest_date_", "")
    return datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S")

In [13]:
def load_valid_folders(start_date,end_date,bucket_name,prefix) :
    valid_folder = {}

    response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix = prefix ,
    Delimiter="/"
    )

    for folder in response.get("CommonPrefixes", []):
        folder_name = folder["Prefix"].split("/")[-2]

        folder = f"{folder_name}/"
        ts = extract_timestamp(folder)

        if ts >= start_date and ts <end_date:
            valid_folder[folder_name]=ts
            print(folder_name)

        else:
            pass

    return valid_folder
    

In [14]:
def load_master_schema():
    with open("master_schema.json", "r") as f:
        schema_dict = json.load(f)
    return schema_dict

def update_scheme_master(column_names, schema_dict):

    columns_added = 0 
    new_columns =[]
    for col in column_names :
        if col not in schema_dict :
            schema_dict[col] = "VARCHAR"
            print("New Column Found : ", col)
            columns_added+=1
            new_columns.append(col)

    if columns_added ==0 :
        pass
    else:
        try:
            json.dumps(schema_dict)
        
            with open("master_schema.json", "w") as f:
                json.dump(schema_dict, f, indent=4)

        except:
            print("Error")            

    return new_columns
#schema_dict= load_master_schema()
#update_scheme_master(column_names,schema_dict)
#schema_dict= load_master_schema()
#print(schema_dict)

In [21]:
def load_data_to_duckdb(valid_folder,Prefix):
    #connect to s3 bucket
    s3 = boto3.client("s3")
    bucket = "nyi-data-analytics"
    
    #connect to DuckDb
    con = duckdb.connect("bronze_staging_db.duckdb")

    #drop the old table (to fetch & clean only the new_data) & Creates One if First Run
    con.execute("DROP TABLE IF EXISTS staging_bronze;")

    files_pushed = 0

    for folder_name, ts in valid_folder.items():

        prefix = f"{Prefix}{folder_name}/"

        # list parquet files
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

        #we will traverse each parquet file , type cast it & upload to duckdb for transformations
        for obj in resp.get("Contents", []):
            key = obj["Key"]
    
            # download file
            file_obj = s3.get_object(Bucket=bucket, Key=key)

            #write file data on temp file
            with tempfile.NamedTemporaryFile(suffix=".parquet") as f:
                f.write(file_obj["Body"].read())
                f.flush()
    
                print("Fetched Data from location : ",key)

                #read columns to type cast them
                schema = con.execute("""
                    DESCRIBE SELECT * FROM read_parquet(?)
                    limit 0
                    """, [f.name]).fetchdf()
                
                column_names = schema["column_name"].tolist()

                schema_dict= load_master_schema()

                new_columns=update_scheme_master(column_names,schema_dict)

                schema_dict= load_master_schema()

                select_expr=[]
                for col in schema_dict.keys():
                    if col in column_names:
                        #column values are present
                        select_expr.append(f'CAST("{col}" AS VARCHAR) AS "{col}"')
                    else:
                        print(f"{col} not present in existing data , values pushed as null")
                        select_expr.append(f'NULL as "{col}"') 
                        
                con.execute(f"""
                CREATE TABLE IF NOT EXISTS staging_bronze AS
                SELECT {select_expr},
                       ? AS ingest_timestamp
                FROM read_parquet(?,union_by_name=true)
                limit 0
                """,[ts,f.name])
                
                con.execute(f"""
                INSERT INTO staging_bronze
                SELECT
                    {select_expr},
                    ? AS ingest_timestamp
                FROM read_parquet(?, union_by_name=true)
                """, [ts, f.name])

                files_pushed += 1

    print(files_pushed)
    return
                   

In [22]:
s3 = boto3.client("s3")

bucket_name = "nyi-data-analytics"
Prefix="bronze/full_load/"
#prefix ko abhi bronze/full_load karna hai

#the dates here are ingestion timestamp
start_date = datetime.strptime("2026-07-01 10:29:00", "%Y-%m-%d %H:%M:%S")
end_date = datetime.strptime("2026-07-03 00:00:00", "%Y-%m-%d %H:%M:%S")

#load valid folders (basically we want to fetch only the data that is required for staging not old data
valid_folder=load_valid_folders(start_date,end_date,bucket_name,Prefix)

load_data_to_duckdb(valid_folder,Prefix)

print()

ingest_date_20260702_091030
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_001.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_002.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_003.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_004.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_005.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_006.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_007.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_008.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_009.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_010.parquet
Fetched Data from location :  bronze/full_load/inges

In [24]:
#Note the count of rows is exactly as same as what we fetched from in the S3 Load
con = duckdb.connect("bronze_staging_db.duckdb")
con.execute("select count(*) from staging_bronze").fetchall()

[(946320,)]